## Ensemble usando imagens originais e geradas pelos descritores fractais

Usando K-fold e regra da soma para aprimorar as predições das redes mobillenet e efficientnet.

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import random
from pathlib import Path
import os


import torch.nn.functional as F

In [3]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
K_FOLDS = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Treinando em:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Treinando em: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Função para carregar o dataset que vem na estrutura
``` plaintext
dataset
    treino_e_validacao
        healthy
            F-RecPlot
                1.png
                2.png
                ...
            originais
                1.tif
                2.tif
                ...
        severe
            F-RecPlot
                ...
            originais
                ...
    testes
        healthy
            ...
        severe
            ...
```
- `dir_data:` o caminho até alguma pasta que contém healthy e severe
- `class_names:` aqui é healthy e severe
- `reshape_type:` a pasta que buscamos, 'originais' ou  'F-RecPlot'

In [5]:
def load_data_from_folders(dir_data, class_names, reshape_type):
    data_list = []
    for class_index, class_name in enumerate(class_names):
        dir_class = Path(dir_data) / class_name / reshape_type

        if dir_class.exists():
            images = list(dir_class.glob('*.*'))
            print(f"Imagens encontradas em {class_name}: {len(images)}")
            for img_path in images:
                data_list.append((str(img_path), class_index))
        else:
            print(f"AVISO: Diretório {dir_class} não encontrado!!!!")

    return data_list

Para preparar o dataset

In [6]:
class ImageDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

Transformação padrão

In [7]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

Primeiro fazemos a função que treina com um único fold, em uma sequência de dados.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_score
)

def train_one_fold(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4
    )

    # Histórico por época
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
        "val_loss": [],
        "val_f1": []
    }

    # Melhores métricas (critério: F1 macro)
    best_metrics = {
        "accuracy": 0.0,
        "f1_macro": 0.0,
        "recall": 0.0,
        "specificity": 0.0,
        "precision": 0.0,
        "balanced_accuracy": 0.0,
        "best_epoch": -1,
        "min_val_loss": float("inf")
    }

    for epoch in range(EPOCHS):
        # ================= TREINO =================
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)

        # ================= VALIDAÇÃO =================
        model.eval()
        val_running_loss = 0.0
        val_preds, val_labels = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * labels.size(0)
                _, preds = torch.max(outputs, 1)

                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        acc = accuracy_score(val_labels, val_preds)
        f1 = f1_score(val_labels, val_preds, average="macro")
        precision = precision_score(
            val_labels, val_preds, pos_label=1, zero_division=0
        )

        cm = confusion_matrix(val_labels, val_preds)
        tn, fp, fn, tp = cm.ravel()

        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        balanced_acc = (recall + specificity) / 2

        history["val_loss"].append(val_loss)
        history["val_f1"].append(f1)
        history["val_accuracy"].append(acc)

        print(
            f"Época {epoch+1}/{EPOCHS} | "
            f"TrainLoss {train_loss:.4f} | "
            f"ValLoss {val_loss:.4f} | "
            f"ValAcc {acc:.4f} | ValF1 {f1:.4f}"
        )

        # ================= MELHOR MODELO =================
        if f1 > best_metrics["f1_macro"]:
            best_metrics.update({
                "accuracy": acc,
                "f1_macro": f1,
                "recall": recall,
                "specificity": specificity,
                "precision": precision,
                "balanced_accuracy": balanced_acc,
                "best_epoch": epoch + 1,
                "min_val_loss": val_loss
            })

    return model, best_metrics, history


Para facilitar a criação dos modelos

In [9]:
def criar_modelo(backbone: str, num_classes: int, pretrained=True):
    backbone = backbone.lower()

    if backbone == "mobilenet":
        model = models.mobilenet_v2(pretrained=pretrained)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes
        )

    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=pretrained)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes
        )

    else:
        raise ValueError("backbone deve ser 'mobilenet' ou 'efficientnet_b0'")

    return model

Essa aqui sim faz o treino com o k folds

In [10]:
def run_kfold(dataset_path, dataset_type, class_names, backbone="mobilenet", seed=SEED):

    os.makedirs(f"models/{seed}", exist_ok=True)

    data_list = load_data_from_folders(dataset_path, class_names, dataset_type)
    print(f"Total de imagens: {len(data_list)}")

    paths = [x[0] for x in data_list]
    labels = [x[1] for x in data_list]

    skf = StratifiedKFold(
        n_splits=K_FOLDS,
        shuffle=True,
        random_state=seed
    )

    fold_results = []
    all_history = []  # <-- histórico COMPLETO (todos os folds)

    best_f1_global = -1
    best_model_state = None

    for fold, (train_idx, val_idx) in enumerate(skf.split(paths, labels)):
        print("\n============================")
        print(f"FOLD {fold+1}/{K_FOLDS}")
        print("============================")

        train_data = [(paths[i], labels[i]) for i in train_idx]
        val_data   = [(paths[i], labels[i]) for i in val_idx]

        train_dataset = ImageDataset(train_data, transform=transform)
        val_dataset   = ImageDataset(val_data, transform=transform)

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=BATCH_SIZE, shuffle=False
        )

        # modelo
        model = criar_modelo(
            backbone=backbone,
            num_classes=len(class_names),
            pretrained=True
        ).to(DEVICE)

        model, metrics, history = train_one_fold(
            model, train_loader, val_loader
        )

        # ---------- melhor modelo global ----------
        if metrics["f1_macro"] > best_f1_global:
            best_f1_global = metrics["f1_macro"]
            best_model_state = model.state_dict()

            nome_modelo = f"models/{seed}/{backbone}_{dataset_type}.pth"
            torch.save(model.state_dict(), nome_modelo)

            print(f"   -> Novo melhor modelo salvo! F1: {best_f1_global:.4f}")

        # ---------- métricas por fold ----------
        fold_results.append({
            "fold": fold + 1,
            **metrics
        })

        # ---------- histórico por época ----------
        all_history.append({
            "fold": fold + 1,
            **history
        })
        
        del model
        torch.cuda.empty_cache()

    # ================= SALVAR RESULTADOS =================
    base_path = f"results_kfold/{seed}/{dataset_type}/{backbone}"
    os.makedirs(base_path, exist_ok=True)

    # métricas finais por fold
    df_metrics = pd.DataFrame(fold_results)
    df_metrics.to_csv(f"{base_path}/metrics.csv", index=False)

    # histórico completo
    df_history = pd.DataFrame(all_history)
    df_history.to_csv(f"{base_path}/history.csv", index=False)

    print("\n===== RESULTADOS FINAIS DO K-FOLD =====")
    print(df_metrics)
    print("\nMédias:")
    print(df_metrics.mean(numeric_only=True))

    # ================= RECRIAR MELHOR MODELO =================
    best_model = criar_modelo(
        backbone=backbone,
        num_classes=len(class_names),
        pretrained=False
    ).to(DEVICE)

    best_model.load_state_dict(best_model_state)

    return best_model


In [11]:
path = 'dataset/treino_e_validacao'
classes = ['healthy', 'severe']
seeds = [42, 65, 121]

results = {}

for seed in seeds:
    print(f"\n===== Treinando com seed {seed} =====")
    set_seed(seed)

    results[seed] = {}

    # MobileNet - F-RecPlot
    results[seed]['mobnet_recplot'] = run_kfold(
        path, "F-RecPlot", classes, backbone="mobilenet", seed=seed
    )
    torch.cuda.empty_cache()

    # EfficientNet-B0 - F-RecPlot
    results[seed]['effnet_recplot'] = run_kfold(
        path, "F-RecPlot", classes, backbone="efficientnet_b0", seed=seed
    )
    torch.cuda.empty_cache()

    # MobileNet - originais
    results[seed]['mobnet_orig'] = run_kfold(
        path, "originais", classes, backbone="mobilenet", seed=seed
    )
    torch.cuda.empty_cache()

    # EfficientNet-B0 - originais
    results[seed]['effnet_orig'] = run_kfold(
        path, "originais", classes, backbone="efficientnet_b0", seed=seed
    )
    torch.cuda.empty_cache()



===== Treinando com seed 42 =====
Imagens encontradas em healthy: 92
Imagens encontradas em severe: 92
Total de imagens: 184

FOLD 1/5


c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainLoss 0.5387 | ValLoss 0.5803 | ValAcc 0.7838 | ValF1 0.7758
Época 2/20 | TrainLoss 0.1802 | ValLoss 0.3975 | ValAcc 0.8649 | ValF1 0.8633
Época 3/20 | TrainLoss 0.0820 | ValLoss 0.2837 | ValAcc 0.8649 | ValF1 0.8633
Época 4/20 | TrainLoss 0.0375 | ValLoss 0.2231 | ValAcc 0.8378 | ValF1 0.8348
Época 5/20 | TrainLoss 0.0139 | ValLoss 0.0921 | ValAcc 0.9730 | ValF1 0.9730
Época 6/20 | TrainLoss 0.0062 | ValLoss 0.0357 | ValAcc 1.0000 | ValF1 1.0000
Época 7/20 | TrainLoss 0.0052 | ValLoss 0.0272 | ValAcc 1.0000 | ValF1 1.0000
Época 8/20 | TrainLoss 0.0025 | ValLoss 0.0248 | ValAcc 1.0000 | ValF1 1.0000
Época 9/20 | TrainLoss 0.0032 | ValLoss 0.0276 | ValAcc 1.0000 | ValF1 1.0000
Época 10/20 | TrainLoss 0.0030 | ValLoss 0.0258 | ValAcc 1.0000 | ValF1 1.0000
Época 11/20 | TrainLoss 0.0205 | ValLoss 0.0212 | ValAcc 1.0000 | ValF1 1.0000
Época 12/20 | TrainLoss 0.0042 | ValLoss 0.0666 | ValAcc 0.9730 | ValF1 0.9729
Época 13/20 | TrainLoss 0.0208 | ValLoss 0.0118 | ValAcc 1.00

c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainLoss 0.5550 | ValLoss 0.5866 | ValAcc 0.7838 | ValF1 0.7798
Época 2/20 | TrainLoss 0.1932 | ValLoss 0.4347 | ValAcc 0.8108 | ValF1 0.8086
Época 3/20 | TrainLoss 0.0861 | ValLoss 0.2901 | ValAcc 0.8649 | ValF1 0.8633
Época 4/20 | TrainLoss 0.0317 | ValLoss 0.1882 | ValAcc 0.9189 | ValF1 0.9187
Época 5/20 | TrainLoss 0.0135 | ValLoss 0.1570 | ValAcc 0.9189 | ValF1 0.9187
Época 6/20 | TrainLoss 0.0075 | ValLoss 0.1641 | ValAcc 0.9189 | ValF1 0.9187
Época 7/20 | TrainLoss 0.0146 | ValLoss 0.1466 | ValAcc 0.9459 | ValF1 0.9459
Época 8/20 | TrainLoss 0.0074 | ValLoss 0.0863 | ValAcc 0.9730 | ValF1 0.9730
Época 9/20 | TrainLoss 0.0012 | ValLoss 0.0574 | ValAcc 0.9730 | ValF1 0.9730
Época 10/20 | TrainLoss 0.0010 | ValLoss 0.0454 | ValAcc 0.9730 | ValF1 0.9730
Época 11/20 | TrainLoss 0.0015 | ValLoss 0.0467 | ValAcc 0.9730 | ValF1 0.9730
Época 12/20 | TrainLoss 0.0075 | ValLoss 0.0708 | ValAcc 0.9730 | ValF1 0.9730
Época 13/20 | TrainLoss 0.0014 | ValLoss 0.0766 | ValAcc 0.97

...

c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainLoss 0.6692 | ValLoss 0.6424 | ValAcc 0.7027 | ValF1 0.7027
Época 2/20 | TrainLoss 0.4693 | ValLoss 0.5654 | ValAcc 0.8378 | ValF1 0.8348
Época 3/20 | TrainLoss 0.3277 | ValLoss 0.4041 | ValAcc 0.8919 | ValF1 0.8912
Época 4/20 | TrainLoss 0.2442 | ValLoss 0.2937 | ValAcc 0.8919 | ValF1 0.8912
Época 5/20 | TrainLoss 0.1461 | ValLoss 0.1946 | ValAcc 0.9459 | ValF1 0.9459
Época 6/20 | TrainLoss 0.1362 | ValLoss 0.1309 | ValAcc 1.0000 | ValF1 1.0000
Época 7/20 | TrainLoss 0.0905 | ValLoss 0.0971 | ValAcc 1.0000 | ValF1 1.0000
Época 8/20 | TrainLoss 0.0526 | ValLoss 0.0763 | ValAcc 1.0000 | ValF1 1.0000
Época 9/20 | TrainLoss 0.0511 | ValLoss 0.0622 | ValAcc 1.0000 | ValF1 1.0000
Época 10/20 | TrainLoss 0.0407 | ValLoss 0.0493 | ValAcc 1.0000 | ValF1 1.0000
Época 11/20 | TrainLoss 0.0243 | ValLoss 0.0427 | ValAcc 1.0000 | ValF1 1.0000
Época 12/20 | TrainLoss 0.0330 | ValLoss 0.0351 | ValAcc 1.0000 | ValF1 1.0000
Época 13/20 | TrainLoss 0.0146 | ValLoss 0.0355 | ValAcc 1.00

c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainLoss 0.6787 | ValLoss 0.7185 | ValAcc 0.5278 | ValF1 0.5245
Época 2/20 | TrainLoss 0.4906 | ValLoss 0.6060 | ValAcc 0.6944 | ValF1 0.6824
Época 3/20 | TrainLoss 0.3507 | ValLoss 0.4948 | ValAcc 0.8056 | ValF1 0.7979
Época 4/20 | TrainLoss 0.2574 | ValLoss 0.3540 | ValAcc 0.9167 | ValF1 0.9161
Época 5/20 | TrainLoss 0.1916 | ValLoss 0.2188 | ValAcc 1.0000 | ValF1 1.0000
Época 6/20 | TrainLoss 0.1565 | ValLoss 0.1346 | ValAcc 1.0000 | ValF1 1.0000
Época 7/20 | TrainLoss 0.0837 | ValLoss 0.0877 | ValAcc 1.0000 | ValF1 1.0000
Época 8/20 | TrainLoss 0.0656 | ValLoss 0.0581 | ValAcc 1.0000 | ValF1 1.0000
Época 9/20 | TrainLoss 0.0423 | ValLoss 0.0443 | ValAcc 1.0000 | ValF1 1.0000
Época 10/20 | TrainLoss 0.0293 | ValLoss 0.0358 | ValAcc 1.0000 | ValF1 1.0000
Época 11/20 | TrainLoss 0.0272 | ValLoss 0.0317 | ValAcc 1.0000 | ValF1 1.0000
Época 12/20 | TrainLoss 0.0234 | ValLoss 0.0284 | ValAcc 1.0000 | ValF1 1.0000
Época 13/20 | TrainLoss 0.0358 | ValLoss 0.0261 | ValAcc 1.00

c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


##### Seção de testes

In [12]:
TEST_DIR = 'dataset/testes'
CLASSES = ['healthy', 'severe']

In [13]:
class EnsembleTestDataset(Dataset):
    def __init__(self, root_dir, class_names, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.class_names = class_names
        self.data = []

        valid_exts = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

        for label_idx, class_name in enumerate(class_names):
            path_orig = self.root / class_name / "originais"
            path_rec  = self.root / class_name / "F-RecPlot"

            if not path_orig.exists() or not path_rec.exists():
                print(f"Diretórios não encontrados para classe {class_name}")
                continue

            # cria um dicionário nome_base -> caminho
            rec_dict = {
                p.stem: p
                for p in path_rec.iterdir()
                if p.is_file() and p.suffix.lower() in valid_exts
            }

            orig_files = [
                p for p in path_orig.iterdir()
                if p.is_file() and p.suffix.lower() in valid_exts
            ]

            for orig_path in orig_files:
                key = orig_path.stem  # nome sem extensão

                if key in rec_dict:
                    self.data.append({
                        "path_orig": str(orig_path),
                        "path_rec": str(rec_dict[key]),
                        "label": label_idx
                    })
                else:
                    print(f"Aviso: Par não encontrado para {orig_path.name}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        img_orig = Image.open(item["path_orig"]).convert("RGB")
        img_rec  = Image.open(item["path_rec"]).convert("RGB")
        label = item["label"]

        if self.transform:
            img_orig = self.transform(img_orig)
            img_rec  = self.transform(img_rec)

        return img_orig, img_rec, label


In [14]:
def carregar_modelo(backbone, num_classes, path_weights):
    print(f"Carregando {backbone} de {path_weights}...")
    backbone = backbone.lower()
    if backbone == "mobilenet":
        model = models.mobilenet_v2(pretrained=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    # Carrega os pesos treinados
    try:
        model.load_state_dict(torch.load(path_weights, map_location=DEVICE))
    except FileNotFoundError:
        print(f"ERRO: Arquivo {path_weights} não encontrado! Treine o modelo antes.")
        return None

    model.to(DEVICE)
    model.eval()
    return model

Criar o dataset e o loader

In [15]:
test_dataset = EnsembleTestDataset(TEST_DIR, CLASSES, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Total de pares de imagens para teste: {len(test_dataset)}")

Total de pares de imagens para teste: 44


In [16]:
def mostrar_metricas(nome_cenario, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0         # Sensibilidade classe positiva
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0    # Classe negativa

    return {
        "acc": acc,
        "f1": f1,
        "recall": recall,
        "specificity": specificity
    }

In [17]:
import pandas as pd
import torch
import torch.nn.functional as F
from collections import defaultdict

all_results = []  # Aqui vamos armazenar todas as métricas para CSV

# Definir os cenários
cenarios = [
    ("MobileNet_Original", "MobileNet_RecPlot"),
    ("MobileNet_Original", "EffNet_Original"),
    ("MobileNet_Original", "EffNet_RecPlot"),
    ("MobileNet_RecPlot",  "EffNet_Original"),
    ("MobileNet_RecPlot",  "EffNet_RecPlot"),
    ("EffNet_Original",    "EffNet_RecPlot")
]

for seed in seeds:

    # Mapear modelos
    models = {
        "MobileNet_Original": results[seed]['mobnet_orig'],
        "MobileNet_RecPlot":  results[seed]['mobnet_recplot'],
        "EffNet_Original":    results[seed]['effnet_orig'],
        "EffNet_RecPlot":     results[seed]['effnet_recplot']
    }

    # Mapear tipo de entrada
    model_input = {
        "MobileNet_Original": "orig",
        "MobileNet_RecPlot":  "rec",
        "EffNet_Original":    "orig",
        "EffNet_RecPlot":     "rec"
    }

    y_true = []

    # Inicializar dicionário para guardar predições
    preds_cenarios = {c: [] for c in cenarios}

    with torch.no_grad():
        for imgs_orig, imgs_rec, labels in test_loader:
            imgs_orig = imgs_orig.to(DEVICE)
            imgs_rec  = imgs_rec.to(DEVICE)

            # Forward de todos os modelos
            outputs = {}
            for name, model in models.items():
                if model_input[name] == "orig":
                    outputs[name] = F.softmax(model(imgs_orig), dim=1)
                else:
                    outputs[name] = F.softmax(model(imgs_rec), dim=1)

            # Guardar labels reais
            y_true.extend(labels.cpu().numpy())

            # Calcular predições para cada cenário
            for cenario in cenarios:
                soma = outputs[cenario[0]] + outputs[cenario[1]]
                y_pred = torch.argmax(soma, dim=1).cpu().numpy()
                preds_cenarios[cenario].extend(y_pred)

    # Calcular métricas e armazenar para CSV
    for cenario, y_pred in preds_cenarios.items():
        # calcular métricas usando sua função mostrar_metricas
        metrics = mostrar_metricas(nome_cenario=" + ".join(cenario), y_true=y_true, y_pred=y_pred)

        metrics["seed"] = seed
        metrics["cenario"] = " + ".join(cenario)

        all_results.append(metrics)

# Criar DataFrame e salvar CSV
df_results = pd.DataFrame(all_results)
df_results.to_csv("resultados_testes.csv", index=False)
